In [ ]:
!pip install -q transformers datasets pandas matplotlib google-generativeai sentencepiece

In [ ]:
import re, time
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import pipeline
import google.generativeai as genai
from getpass import getpass
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
data = load_dataset("gsm8k", "main")["test"].select(range(2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
hf = pipeline("text-generation", model="distilgpt2")
gm = genai.GenerativeModel("gemini-2.5-flash")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def zero(q): return f"Q: {q}\nA:"
def few(q): return f"Q: 2+3?\nA: 5\nQ: 10-4?\nA: 6\nQ: {q}\nA:"
def cot(q): return f"Solve step by step.\nQ: {q}\nA:"

In [ ]:
  def ask_hf(p):
    try:
      return hf(p, max_new_tokens=40, do_sample=False, pad_token_id=50256)[0]["generated_text"]
    except:
      return "HF Error"

  def ask_gm(p):
    try:
      return gm.generate_content(p).text
    except:
      return "Quota exceeded / Try later"

  def get_num(x):
    n = re.findall(r"-?\d+\.?\d*", str(x).replace(",", ""))
    return n[-1] if n else None

  def get_gold(x):
      m = re.search(r"####\s*(-?\d+\.?\d*)", x.replace(",", ""))
      return m.group(1) if m else get_num(x) # Result

  def check(pred, gold):
      if pred is None:
        return "Hallucination"
      elif pred == gold:
        return "Correct"
      else:
        return "Wrong"

In [ ]:
res = []
for r in data:
  q = r["question"]
  gold = get_gold(r["answer"])

  for name, fn in [("Zero-shot", zero), ("Few-shot", few), ("CoT", cot)]:
    p = fn(q)

    hf_out = ask_hf(p)
    gm_out = ask_gm(p)
    time.sleep(10)

    hf_pred = get_num(hf_out)
    gm_pred = get_num(gm_out)

    res.append([name, q[:50], gold, hf_pred, gm_pred, check(hf_pred, gold), check(gm_pred, gold)])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4037.50ms
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will tak

In [ ]:
df = pd.DataFrame(res, columns=["Prompt", "Question", "Gold", "HF_Pred", "Gemini_Pred", "HF_Result", "Gemini_Result"])
print(df)

      Prompt                                           Question Gold HF_Pred  \
0  Zero-shot  Janet’s ducks lay 16 eggs per day. She eats th...   18       2   
1   Few-shot  Janet’s ducks lay 16 eggs per day. She eats th...   18       5   
2        CoT  Janet’s ducks lay 16 eggs per day. She eats th...   18       2   
3  Zero-shot  A robe takes 2 bolts of blue fiber and half th...    3     1.5   
4   Few-shot  A robe takes 2 bolts of blue fiber and half th...    3      -4   
5        CoT  A robe takes 2 bolts of blue fiber and half th...    3     1.5   

  Gemini_Pred HF_Result  Gemini_Result  
0          18     Wrong        Correct  
1          18     Wrong        Correct  
2        None     Wrong  Hallucination  
3        None     Wrong  Hallucination  
4           3     Wrong        Correct  
5        None     Wrong  Hallucination  


In [ ]:
acc = pd.DataFrame({ "HF": df["HF_Result"].eq("Correct").groupby(df["Prompt"]).mean() * 100, "Gemini": df["Gemini_Result"].eq("Correct").groupby(df["Prompt"]).mean() * 100 })
print("\nAccuracy (%)")
print(acc)


Accuracy (%)
            HF  Gemini
Prompt                
CoT        0.0     0.0
Few-shot   0.0   100.0
Zero-shot  0.0    50.0


In [ ]:
print("\nHF Summary")
print(df["HF_Result"].value_counts())
print("\nGemini Summary")
print(df["Gemini_Result"].value_counts())


HF Summary
HF_Result
Wrong    6
Name: count, dtype: int64

Gemini Summary
Gemini_Result
Correct          3
Hallucination    3
Name: count, dtype: int64
